Tải dữ liệu

In [2]:
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
import os
import pandas as pd # Thêm thư viện pandas

# --- 1. Định nghĩa đường dẫn ---
image_dir = r"D:\Python_mohinh\data\mel_spectrograms_db_30s"
csv_path = r"D:\Python_mohinh\data\mel_metadata_new_30s.csv"
# Chữ 'r' đứng trước chuỗi để xử lý dấu '\' của Windows một cách chính xác

# --- 2. Đọc CSV và tạo danh sách file/nhãn ---
df = pd.read_csv(csv_path)

# Lấy danh sách tên class (genre) duy nhất và sắp xếp
# Điều này rất quan trọng để ánh xạ 'Hip-Hop' -> 0, 'Pop' -> 1, v.v.
class_names = sorted(list(df['genre'].unique()))
label_map = {name: i for i, name in enumerate(class_names)}

# Tạo danh sách đường dẫn ảnh đầy đủ và nhãn (dưới dạng số)
image_paths = []
labels = []

for index, row in df.iterrows():
    filename = row['filename']
    genre = row['genre']
    
    # Tạo đường dẫn đầy đủ tới file ảnh
    full_path = os.path.join(image_dir, filename)
    
    # Chỉ thêm vào danh sách nếu file thực sự tồn tại
    if os.path.exists(full_path):
        image_paths.append(full_path)
        labels.append(label_map[genre])
    else:
        # Cảnh báo nếu file trong CSV không có ảnh tương ứng
        print(f"Cảnh báo: Không tìm thấy file {full_path}")

print(f"Đã tìm thấy {len(image_paths)} ảnh hợp lệ.")
print(f"Các lớp (thể loại): {class_names}")

# --- 3. Phân chia Train / Val / Test (Giữ nguyên) ---
# Dùng 'stratify=labels' để đảm bảo các tập train/val/test có tỷ lệ thể loại nhạc như nhau
# Chia lần 1: Tách Test set (ví dụ 15%)
train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
    image_paths, labels, test_size=0.15, random_state=42, stratify=labels)

# Chia lần 2: Tách Train và Validation từ phần còn lại
train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_val_paths, train_val_labels, test_size=(0.15/0.85), random_state=42, stratify=train_val_labels) # (15% / (100% - 15%))

# --- 4. Tạo hàm tải và tiền xử lý ảnh (Giữ nguyên) ---
# Đảm bảo kích thước này khớp với ảnh bạn đã trích xuất
IMG_HEIGHT = 128
IMG_WIDTH = 1292 
NUM_CHANNELS = 3 # Mô hình pre-trained yêu cầu 3 kênh
NUM_CLASSES = len(class_names) # Cập nhật số lớp tự động từ CSV

def load_and_preprocess_image(path, label):
    image = tf.io.read_file(path)
    # Sửa lỗi: Dùng decode_jpeg thay vì decode_image
    # channels=1 đảm bảo ảnh được đọc là 1 kênh (grayscale)
    image = tf.image.decode_jpeg(image, channels=1) 
    
    # Dòng này bây giờ sẽ hoạt động
    image = tf.image.resize(image, [IMG_HEIGHT, IMG_WIDTH])
    
    # Các dòng còn lại giữ nguyên
    image = tf.image.grayscale_to_rgb(image) 
    image = tf.cast(image, tf.float32)
    return image, label

# --- 5. Tạo tf.data.Dataset (Giữ nguyên) ---
# Hàm này không đổi
def create_dataset(paths, labels):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(32) # Điều chỉnh batch size (ví dụ: 16, 32) tùy theo VRAM
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

# Tạo các bộ dữ liệu
train_ds = create_dataset(train_paths, train_labels)
val_ds = create_dataset(val_paths, val_labels)
test_ds = create_dataset(test_paths, test_labels)

print("\n--- Hoàn tất chuẩn bị dữ liệu ---")
print(f"Tổng số lớp: {NUM_CLASSES}")
print(f"Kích thước tập Train: {len(train_paths)}")
print(f"Kích thước tập Val:   {len(val_paths)}")
print(f"Kích thước tập Test:    {len(test_paths)}")

Đã tìm thấy 7994 ảnh hợp lệ.
Các lớp (thể loại): ['Electronic', 'Experimental', 'Folk', 'Hip-Hop', 'Instrumental', 'International', 'Pop', 'Rock']

--- Hoàn tất chuẩn bị dữ liệu ---
Tổng số lớp: 8
Kích thước tập Train: 5595
Kích thước tập Val:   1199
Kích thước tập Test:    1200


Xây dựng Mô hình (Transfer Learning)

In [3]:
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.applications import EfficientNetV2B0

# --- 1. Tải Base Model ---
# Đầu vào là (128, 1292, 3)
# Chúng ta không cần resize về (224, 224) vì GlobalAveragePooling2D
# có thể xử lý kích thước bất kỳ.
base_model = EfficientNetV2B0(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, NUM_CHANNELS),
    include_top=False, # Bỏ lớp 1000-class của ImageNet
    weights='imagenet'
)

# --- 2. Đóng băng Base Model ---
base_model.trainable = False

# --- 3. Thêm các lớp của chúng ta ---
inputs = Input(shape=(IMG_HEIGHT, IMG_WIDTH, NUM_CHANNELS))

# Thêm lớp tiền xử lý của EfficientNet (chuẩn hóa pixel)
# Bạn cũng có thể thêm Augmentation ở đây
x = tf.keras.applications.efficientnet_v2.preprocess_input(inputs)

x = base_model(x, training=False) # Chạy base model ở chế độ inference
x = GlobalAveragePooling2D()(x) # Bước này gom đặc trưng lại
x = Dropout(0.3)(x) # Giảm overfitting
x = Dense(128, activation='relu')(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x) # 8 thể loại

model = Model(inputs, outputs)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 128, 1292, 3)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-b0 (Functional)  │ (None, 4, 41, 1280)    │     5,919,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │         1,032 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,084,312 (23.21 MB)

 Trainable params: 165,000 (644.53 KB)

 Non-trainable params: 5,919,312 (22.58 MB)

Huấn luyện (Training) - Giai đoạn 1: Huấn luyện "Đầu"

In [4]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy', # Vì nhãn là số (0-7)
    metrics=['accuracy']
)

# Huấn luyện chỉ các lớp "đầu" (đã đóng băng base model)
history = model.fit(
    train_ds,
    epochs=15, # Huấn luyện trong 15 epochs
    validation_data=val_ds
)

Epoch 1/15
148/175 ━━━━━━━━━━━━━━━━━━━━ 45s 2s/step - accuracy: 0.3366 - loss: 1.7931

KeyboardInterrupt: 

Tinh chỉnh (Fine-Tuning) - Giai đoạn 2: Tăng cường

In [ ]:
# --- 1. Mở băng Base Model ---
base_model.trainable = True

# Ta chỉ nên mở băng các lớp cuối (ví dụ 30 lớp cuối)
# Huấn luyện toàn bộ có thể gây overfitting
for layer in base_model.layers[:-30]:
    layer.trainable = False

# --- 2. Compile lại với Learning Rate RẤT THẤP ---
# Rất quan trọng, nếu LR cao sẽ phá hỏng trọng số pre-trained
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), # 0.00001
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# --- 3. Huấn luyện tiếp (Fine-tuning) ---
fine_tune_epochs = 10
total_epochs = 15 + fine_tune_epochs

history_fine_tune = model.fit(
    train_ds,
    epochs=total_epochs,
    initial_epoch=history.epoch[-1], # Bắt đầu từ epoch đã dừng
    validation_data=val_ds
)

Đánh giá và Trực quan hóa

In [ ]:
loss, accuracy = model.evaluate(test_ds)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Lấy dự đoán
y_pred = []
y_true = []
for images, labels in test_ds:
    y_pred.extend(np.argmax(model.predict(images), axis=1))
    y_true.extend(labels.numpy())

# Vẽ ma trận
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
plt.figure(figsize=(15, 10))
# Lấy 1 batch ảnh từ train_ds
images, labels = next(iter(train_ds)) 

for i in range(8): # Hiển thị 8 ảnh
    ax = plt.subplot(4, 2, i + 1)
    # Vì ảnh đã chuẩn hóa cho model, ta cần scale về [0, 255]
    img_to_show = images[i].numpy().astype("uint8")

    plt.imshow(img_to_show)
    plt.title(f"Genre: {class_names[labels[i].numpy()]}")
    plt.axis("off")
plt.show()

In [4]:
import numpy as np
import os

# --- CẤU HINH ---
# 1. Đặt đường dẫn cơ sở của bạn (giống trong Cell 1)
NPY_BASE_DIR = r'D:\Python_mohinh\data\audio_features_npy_seq'

# 2. File NPY bạn muốn kiểm tra (đã lấy từ lỗi của bạn)
TEN_FILE_NPY = '000002_chunk_0.npy' 

# 3. Tên file text bạn muốn lưu kết quả
OUTPUT_TEXT_FILE = 'npy_content.txt'
# -----------------

file_path = os.path.join(NPY_BASE_DIR, TEN_FILE_NPY)

try:
    # Tải file .npy
    data = np.load(file_path, allow_pickle=True)
    
    print(f"--- Đã tải file: {TEN_FILE_NPY} ---")
    print(f"Hình dạng (Shape) của mảng là: {data.shape}")

    # --- ĐÃ SỬA LỖI ---
    # np.savetxt() tự động lưu toàn bộ. Chỉ cần bỏ 'threshold=np.inf'
    np.savetxt(OUTPUT_TEXT_FILE, data, fmt='%.8f') 
    
    print(f"\n✅ THÀNH CÔNG!")
    print(f"Đã lưu TOÀN BỘ nội dung (hình dạng {data.shape}) vào file: {OUTPUT_TEXT_FILE}")
    print("Bạn có thể mở file này từ thanh File Explorer bên trái để xem.")

except FileNotFoundError:
    print(f"LỖI: Không tìm thấy file tại đường dẫn: {file_path}")
except Exception as e:
    print(f"Lỗi khi xử lý file: {e}")

--- Đã tải file: 000002_chunk_0.npy ---
Hình dạng (Shape) của mảng là: (431, 57)

✅ THÀNH CÔNG!
Đã lưu TOÀN BỘ nội dung (hình dạng (431, 57)) vào file: npy_content.txt
Bạn có thể mở file này từ thanh File Explorer bên trái để xem.
